# Assignment 4.3: Attention

You will write scaled dot-product attention, the core of the transformer architecture, and use it to
look inside BERT, a pretrained transformer. A multi-head attention layer built on your function is
provided. You will run BERT on two sentences, one given and one of your own: first to watch it fill
in a hidden word, the task it was trained on, then to draw the attention of its heads. Your layer
runs three ways on each sentence: with random weights, with the weights of BERT's first layer, and
with the weights of a later BERT layer.

Three code cells are yours: `scaled_dot_product_attention`, your sentence, and the BERT layer you
choose. Everything else is provided and already runs. Nothing is trained in this notebook.

# Running in Colab

Open this notebook from its Colab link in the spec, then save your own copy right away with
**File > Save a copy in Drive**. Work in that copy, or your changes are lost when Colab closes. This
notebook needs no data files and no GPU.

# Import libraries
Do not use any other Python library.

torch - PyTorch, the neural network library. `torch.nn` holds the linear layers and `torch.nn.functional` the softmax.

transformers - Hugging Face library from which we load the pretrained BERT model and its tokenizer.

matplotlib - Graphing library for visualizing results.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMaskedLM
import matplotlib.pyplot as plt

# What BERT does

BERT was pretrained on about 3.3 billion words of English text: BookCorpus, a collection of
unpublished books, and English Wikipedia. Its training task is masked word prediction. About 15% of
the tokens in each training sentence are hidden behind a `[MASK]` token, and BERT learns to predict
each hidden word from the words on both sides of it. (It was also trained to predict whether one
sentence follows another.) Nothing in that task tells BERT which words relate to which; whatever
patterns its attention heads show, they learned because those patterns helped predict hidden words.

# Load BERT

Loads the pretrained `bert-base-uncased` model with its masked word prediction head, and its
tokenizer. The first run downloads about 440 MB from Hugging Face and keeps it in a local cache.
`bert` is the transformer itself: 12 layers, each with 12 attention heads over a model dimension of
768. The prediction head on top turns BERT's output at a `[MASK]` position into a score for every
word in its vocabulary. A warning that some checkpoint weights were not used is expected: they
belong to the next-sentence task, which this notebook does not use.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
mlm = AutoModelForMaskedLM.from_pretrained("bert-base-uncased", attn_implementation="eager")
mlm.eval()
bert = mlm.bert

# Function: scaled_dot_product_attention

Computes attention for every head at once:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

The softmax runs over the keys, the last dimension, so each row of the attention matrix sums to 1.
Where the mask is 0 (padding), the score is set to a large negative number before the softmax, so
that position gets no attention. That line is provided. Use `torch.matmul`, `.transpose(-2, -1)`,
and `F.softmax`.

Inputs:
*   **Q**, **K**, **V**: tensors of shape (B, H, L, d_k), where B is the batch size, H the number of heads, L the sequence length, and d_k the size of each head
*   **mask**: tensor broadcastable to (B, H, L, L), 1 to keep a position and 0 to ignore it, or None

Outputs:
*   **out**: tensor of shape (B, H, L, d_k), the attention-weighted values
*   **attn**: tensor of shape (B, H, L, L), the attention matrix of each head

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    # 1. raw scores: each query against each key, scaled by the square root of d_k
    # TODO
    scores = None

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    # 2. attention matrix: softmax over the keys dimension
    # 3. output: the attention matrix applied to the values
    # TODO
    attn, out = None, None
    return out, attn

# Class: MultiHeadAttention

$$\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)W^O, \qquad \text{head}_i = \text{Attention}(XW_i^Q, XW_i^K, XW_i^V)$$

This class is provided. Its four linear layers are each d_model by d_model: `W_q`, `W_k`, and `W_v`
project the input into queries, keys, and values for all heads at once, and `W_o` mixes the heads'
outputs back together.

`forward` projects the input with `W_q`, `W_k`, and `W_v`, then splits the last dimension into heads:
it reshapes each from (B, L, d_model) to (B, L, H, d_k) and swaps the L and H dimensions. It calls
your `scaled_dot_product_attention` on all heads at once, merges the heads back into one
(B, L, d_model) tensor, and applies `W_o`.

Inputs to `forward`:
*   **x**: tensor of shape (B, L, d_model)
*   **mask**: as in `scaled_dot_product_attention`

Outputs:
*   **out**: tensor of shape (B, L, d_model)
*   **attn**: tensor of shape (B, H, L, L)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=768, num_heads=12):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must split evenly"
        self.h = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, L, C = x.shape
        # project, then split the last dimension into heads
        Q = self.W_q(x).view(B, L, self.h, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, L, self.h, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, L, self.h, self.d_k).transpose(1, 2)
        out, attn = scaled_dot_product_attention(Q, K, V, mask)
        # merge the heads and mix them
        out = out.transpose(1, 2).contiguous().view(B, L, C)
        out = self.W_o(out)
        return out, attn

# Function: embed_sentence

This function is provided. It tokenizes a sentence and runs it through BERT once. It returns BERT's
token embeddings (the input to its first layer), the hidden states that enter every layer, the
attention matrices BERT itself computed in every layer, the padding mask, and the tokens as text.
BERT adds a `[CLS]` token at the start and a `[SEP]` token at the end.

# Function: load_bert_layer

This function is provided. It builds one of your `MultiHeadAttention` layers and copies into it the
query, key, value, and output weights of one BERT layer.

# Function: plot_heads

This function is provided. It draws the attention matrix of every head as a heatmap. Row i shows how
much token i attends to each token. Darker means less attention.

# Function: fill_mask

This function is provided. It hides one word of a sentence behind `[MASK]`, runs BERT with its
prediction head, and prints the five words BERT finds most likely in that position, with their
probabilities.

In [ ]:
@torch.no_grad()
def embed_sentence(text):
    ids = tokenizer(text, return_tensors="pt")
    outputs = bert(**ids, output_hidden_states=True, output_attentions=True)
    mask = ids["attention_mask"].unsqueeze(1).unsqueeze(2)
    tokens = tokenizer.convert_ids_to_tokens(ids["input_ids"][0])
    return outputs.hidden_states, outputs.attentions, mask, tokens


@torch.no_grad()
def load_bert_layer(layer_id):
    layer = bert.encoder.layer[layer_id].attention
    mha = MultiHeadAttention(d_model=768, num_heads=12)
    mha.W_q.load_state_dict(layer.self.query.state_dict())
    mha.W_k.load_state_dict(layer.self.key.state_dict())
    mha.W_v.load_state_dict(layer.self.value.state_dict())
    mha.W_o.load_state_dict(layer.output.dense.state_dict())
    return mha.eval()


def plot_heads(attn, tokens, title, heads_per_row=4):
    attn = attn[0].detach().cpu()
    H = attn.shape[0]
    rows = (H + heads_per_row - 1) // heads_per_row
    fig, axes = plt.subplots(rows, heads_per_row, figsize=(4 * heads_per_row, 4 * rows))
    axes = axes.flatten()
    for h in range(H):
        axes[h].imshow(attn[h], vmin=0, vmax=float(attn.max()), cmap="viridis")
        axes[h].set_xticks(range(len(tokens)), tokens, rotation=90, fontsize=8)
        axes[h].set_yticks(range(len(tokens)), tokens, fontsize=8)
        axes[h].set_title(f"Head {h}")
    for ax in axes[H:]:
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


@torch.no_grad()
def fill_mask(text, word, top_k=5):
    words = text.split()
    # hide the first occurrence of word, ignoring case and punctuation
    i = next((i for i, w in enumerate(words) if w.strip(".,!?;:\"'").lower() == word.lower()), None)
    if i is None:
        raise ValueError(f"{word!r} is not a word of the sentence")
    words[i] = words[i].lower().replace(word.lower(), tokenizer.mask_token)
    masked = " ".join(words)
    ids = tokenizer(masked, return_tensors="pt")
    position = (ids["input_ids"][0] == tokenizer.mask_token_id).nonzero()[0].item()
    probabilities = F.softmax(mlm(**ids).logits[0, position], dim=-1)
    top = probabilities.topk(top_k)
    print(masked)
    for p, token_id in zip(top.values, top.indices):
        print(f"  {tokenizer.decode([int(token_id)]):12} {p.item():.3f}")

# The two sentences

The given sentence is a classic test of whether a model links a pronoun to the right noun: "it"
refers to the trophy, because the trophy is what was too big. BERT will be asked to fill in "big".

Write your own sentence of 8 to 20 words, and choose one of its words for BERT to fill in. Pick a
sentence where some words clearly refer to other words, for example a pronoun and the noun it stands
for, so the attention patterns have something to show. Choose a word that the rest of the sentence
gives clues about.

In [ ]:
GIVEN_SENTENCE = "The trophy did not fit in the suitcase because it was too big."
GIVEN_MASK_WORD = "big"

# 1. a sentence of your own, 8 to 20 words, and one of its words for bert to fill in
# TODO
SENTENCE, MASK_WORD = None, None

SENTENCES = {"given": GIVEN_SENTENCE, "yours": SENTENCE}
embedded = {name: embed_sentence(text) for name, text in SENTENCES.items()}
for name, (_, _, _, tokens) in embedded.items():
    print(name, tokens)

# Fill in the blank

BERT's own task: each sentence with one word hidden, and BERT's five best guesses for it.

In [ ]:
fill_mask(GIVEN_SENTENCE, GIVEN_MASK_WORD)
print()
fill_mask(SENTENCE, MASK_WORD)

# Untrained attention

Your layer with random weights, run on BERT's token embeddings of each sentence. Nothing has been
learned yet.

In [ ]:
torch.manual_seed(0)
untrained = MultiHeadAttention(d_model=768, num_heads=12).eval()
for name, (hidden_states, bert_attentions, mask, tokens) in embedded.items():
    with torch.no_grad():
        _, attn_untrained = untrained(hidden_states[0], mask)
    plot_heads(attn_untrained, tokens, f"Untrained multi-head attention, {name} sentence")

# BERT layer 0

Your layer with the weights of BERT's first layer copied in, run on each sentence. If your
implementation is right, its attention matrices are the ones BERT computed itself, and the printed
difference is close to 0.

In [ ]:
mha_0 = load_bert_layer(0)
for name, (hidden_states, bert_attentions, mask, tokens) in embedded.items():
    with torch.no_grad():
        _, attn_0 = mha_0(hidden_states[0], mask)
    plot_heads(attn_0, tokens, f"BERT layer 0, {name} sentence, computed by your MultiHeadAttention")
    print(f"{name} sentence: largest difference from BERT's own layer 0 attention: "
          f"{(attn_0 - bert_attentions[0]).abs().max().item():.2e}")

# A later BERT layer

Choose a later layer, from 1 to 11, and run your layer with its weights. A layer's input is the
output of the layer before it, so this cell feeds your layer the hidden state that enters the layer you
choose.

In [ ]:
# 1. a later bert layer, 1 to 11
# TODO
LAYER_ID = None

mha_later = load_bert_layer(LAYER_ID)
for name, (hidden_states, bert_attentions, mask, tokens) in embedded.items():
    with torch.no_grad():
        _, attn_later = mha_later(hidden_states[LAYER_ID], mask)
    plot_heads(attn_later, tokens, f"BERT layer {LAYER_ID}, {name} sentence, computed by your MultiHeadAttention")
    print(f"{name} sentence: largest difference from BERT's own layer {LAYER_ID} attention: "
          f"{(attn_later - bert_attentions[LAYER_ID]).abs().max().item():.2e}")

# Report

Cover the following in your report.

*   BERT's guesses for the hidden word in each sentence. Say whether they were sensible and which words of the sentence give the clues.
*   The heatmaps of all three runs (random weights, BERT layer 0, and your later layer) on both sentences, and the printed differences.
*   The patterns of at least three heads in layer 0 and in your later layer. Compare them with each other and with the untrained heads.

Patterns to look for include a head that attends to the previous or next token, a head that attends to `[CLS]` or `[SEP]`, and a head that links words that refer to each other, such as "it" and "trophy" in the given sentence.